In [1]:
import pandas as pd 
import numpy as np 

In [2]:
ls "../../textos"

christenson-english-MOD.txt  osu-transcription-flat.txt
christenson-english.txt      osu-transcription-notes.txt
christenson-literal.txt      osu-transcription.pdf
colop-glossary.txt           recinos-mod.txt
colop-pk.txt                 recinos.txt
edmonson-1971.pdf            tedlock-text-only.txt
facsimiles/                  textos.zip
morely.txt                   ximenez-historia.txt
newberry-record.txt          xml/
osu-transcription-cols.txt


In [3]:
src_file = "../../textos/christenson-english-MOD.txt"
lines = open(src_file, "r").read().split("\n\n")

In [4]:
page_num_pat = r"^\d+\s*$"
# footnote_pat = r"^\s*\d+"
chap_name_pat = r"^\s*[A-Z][A-Z ]+\d*\s*$"
start_page_num = 49

In [5]:
LINES = pd.DataFrame(lines)
LINES.columns = ['line_str']
LINES.index.name = 'line_num'

# Handle Pages (why tho?)
# LINES.loc[LINES.line_str.str.match(page_num_pat), 'page_num'] = LINES.line_str
# LINES.page_num = LINES.page_num.ffill().fillna('0')
# LINES = LINES.loc[~LINES.line_str.str.match(page_num_pat)].copy()
# LINES.page_num = LINES.page_num.astype(int)

# Handle Chapters
LINES.loc[LINES.line_str.str.match(chap_name_pat), 'chap_name'] = LINES.line_str
LINES.chap_name = LINES.chap_name.str.strip()
LINES.chap_name = LINES.chap_name.str.replace(r"\d+", "", regex=True)
CHAPS = LINES.dropna()[['chap_name']]
CHAPS.index = [i+1 for i in range(len(CHAPS))]
CHAPS.index.name = 'chap_num'
LINES['chap_num'] = LINES.chap_name.map(CHAPS.reset_index().set_index('chap_name').chap_num)
LINES.chap_num = LINES.chap_num.ffill()
LINES.chap_num = LINES.chap_num.astype(int)
LINES = LINES.drop('chap_name', axis=1)
LINES = LINES.reset_index().set_index(['chap_num','line_num'])
LINES = LINES[~LINES.line_str.str.match(chap_name_pat)].copy()
LINES.line_str = LINES.line_str.str.strip().str.replace(r"\d+", "", regex=True)
LINES

line_str
chap_num line_num                                                   
1        1         THIS IS THE BEGINNING OF THE ANCIENT TRADITION...
         2                                             called Quiche
         3         HERE we shall write  We shall begin to tell th...
         4         the origin of all that was done in the citadel...
         5                                                    nation
...                                                              ...
86       3589      there are three stewards, one before each of t...
         3590                                                       
         3591      But this is the essence of the Quiches, becaus...
         3592      seeing it. It was with the lords at first, but...
         3593      There is only this. All is now completed conce...

[3508 rows x 1 columns]

In [6]:
CHAPS['chap_str'] = LINES.groupby('chap_num').line_str.apply(lambda x: " ".join(x))

In [7]:
CHAPS

,chap_name,chap_str
chap_num,,
1,PREAMBLE,THIS IS THE BEGINNING OF THE ANCIENT TRADITION...
2,THE PRIMORDIAL WORLD,THIS IS THE ACCOUNT of when all is still silen...
3,THE CREATION OF THE EARTH,THEN came his word. Heart of Sky arrived here ...
4,THE CREATION OF THE ANIMALS,THEN were conceived the animals of the mountai...
5,THE FALL OF THE ANIMALS,THEN it was said to the deer and the birds by ...
...,...,...
82,THE DYNASTY OF NIHAIB LORDS,"THESE, then, are the nine highest of the great..."
83,THE GREAT HOUSES OF THE NIHAIB LORDS,"THESE, then, are all the lords who follow behi..."
84,THE DYNASTY OF AHAU QUICHE LORDS,"THESE, then, are they of the Ahau Quiches: Mah..."


In [8]:
DOC = CHAPS[['chap_str']].rename(columns={'chap_str':'doc_str'})
DOC

,doc_str
chap_num,
1,THIS IS THE BEGINNING OF THE ANCIENT TRADITION...
2,THIS IS THE ACCOUNT of when all is still silen...
3,THEN came his word. Heart of Sky arrived here ...
4,THEN were conceived the animals of the mountai...
5,THEN it was said to the deer and the birds by ...
...,...
82,"THESE, then, are the nine highest of the great..."
83,"THESE, then, are all the lords who follow behi..."
84,"THESE, then, are they of the Ahau Quiches: Mah..."


In [9]:
slug = "christenson_english_prose"
CHAPS.to_csv(f"{slug}-CHAP.csv", index=True)
DOC.to_csv(f"{slug}-DOC.csv", index=True)
with open(f"{slug}-DOC_idx.txt", "w") as outfile:
    outfile.write(",".join(DOC.index.names))